In [19]:
using SumOfSquares
using DynamicPolynomials
using JuMP
using MultivariatePolynomials
using LinearAlgebra
using Random, Combinatorics
using SCS
using DataFrames
using CSV

const MOI = JuMP.MOI
const DEFAULT_SOLVER = "SCS"
# using MosekTools
# const DEFAULT_SOLVER = "MOSEK"

"SCS"

Helper functions for submodular regression

In [20]:
function optimizer_factory(solver_kind::AbstractString = DEFAULT_SOLVER)
    normalized = uppercase(strip(solver_kind))
    if normalized == "SCS"
        return optimizer_with_attributes(SCS.Optimizer, "verbose" => 0)
    elseif normalized == "MOSEK"
        configure_mosek_license!()
        return optimizer_with_attributes(Mosek.Optimizer)
    end
end

function make_model(; sos::Bool, solver_kind::AbstractString = DEFAULT_SOLVER)
    optimizer = optimizer_factory(solver_kind)
    return sos ? SOSModel(optimizer) : Model(optimizer)
end

function multilinear_monomials(x, d, lowest_deg = 1)
    n = length(x)
    monoms = Any[1]
    for degree in lowest_deg:d
        for combo in combinations(1:n, degree)
            push!(monoms, prod(x[idx] for idx in combo))
        end
    end
    return monoms
end

function get_second_derivative(f, variables, i, j)
    return differentiate(differentiate(f, variables[i]), variables[j])
end

function build_multilin_var(model, x, monom, name)
    w = @variable(model, [monom], base_name = name)
    f = sum(w[m] * m for m in monom)
    return f, w
end

function f_sos_submod(model, x, f, t)
    n = length(x)
    for i in 1:n
        for j in i+1:n
            d2f = get_second_derivative(f, x, i, j)
            domain = algebraicset([xi^2 - xi for xi in x if (xi != x[i]) && (xi != x[j])])
            @constraint(model, -d2f >= 0, domain = domain, maxdegree = 2 * t)
        end
    end
end

function eval_f_indicator(f, n, idxs)
    vals = zeros(n)
    vals[idxs] .= 1.0
    return f(vals)
end

function tot_loss_setwise(f, n, data)
    total = 0.0
    for (S, y) in data
        total += (eval_f_indicator(f, n, S) - y)^2
    end
    return total
end

function monomial_supports(monoms)
    return [m == 1 ? Int[] : findall(!iszero, exponents(m)) for m in monoms]
end

function build_feature_matrix(data, monoms, n)
    supports = monomial_supports(monoms)
    m = length(data)
    p = length(monoms)
    A = Matrix{Float64}(undef, m, p)
    y = Vector{Float64}(undef, m)
    present = falses(n)

    for r in 1:m
        S, yr = data[r]
        fill!(present, false)
        present[S] .= true
        y[r] = yr

        for j in 1:p
            val = 1.0
            for idx in supports[j]
                if !present[idx]
                    val = 0.0
                    break
                end
            end
            A[r, j] = val
        end
    end

    return A, y
end

function solver_stats(model)
    return (
        solver = solver_name(model),
        run_time = try
            MOI.get(backend(model), MOI.SolveTimeSec())
        catch
            NaN
        end,
        termination_status = string(termination_status(model)),
        primal_status = string(primal_status(model)),
        dual_status = string(dual_status(model)),
    )
end

function test_error(f, n, data)
    total = 0.0
    for (S, y) in data
        total += (eval_f_indicator(f, n, S) - y)^2
    end
    return total / length(data)
end

function materialize_multilinear_solution(wf, monoms)
    return sum(value(wf[m]) * m for m in monoms)
end

function materialize_poly_solution(wf, monoms)
    return sum(value(wf[i]) * monoms[i] for i in 1:length(monoms))
end

function regression_row(dataset, label, res, fitted_f, n, train_data, val_data, test_data)
    return (
        dataset = dataset,
        model = label,
        train_mse = test_error(fitted_f, n, train_data),
        val_mse = test_error(fitted_f, n, val_data),
        test_mse = test_error(fitted_f, n, test_data),
        solve_time = res.time,
        solver = res.solver,
        termination_status = res.termination_status,
    )
end

function read_split_csv(path::AbstractString; one_based::Bool = false)
    df = CSV.read(path, DataFrame; normalizenames = true)
    data = Vector{Tuple{Vector{Int}, Float64}}(undef, nrow(df))
    for (idx, row) in enumerate(eachrow(df))
        raw = row[:subset]
        subset_string = raw === missing ? "" : strip(String(raw))
        S = isempty(subset_string) ? Int[] : parse.(Int, split(subset_string))
        if !one_based
            S .= S .+ 1
        end
        data[idx] = (S, Float64(row[:value]))
    end
    return data
end

infer_n(data::Vector{Tuple{Vector{Int}, Float64}}) = isempty(data) ? 0 : maximum([isempty(S) ? 0 : maximum(S) for (S, _) in data])

function dataset_root(repo_root::AbstractString = pwd())
    candidates = [
        joinpath(repo_root, "regression", "synthetic_data"),
        joinpath(repo_root, "synthetic_data"),
    ]
    for root in candidates
        if isdir(root)
            return root
        end
    end
    error("missing bundled data directory; upload either `regression/synthetic_data` or `synthetic_data` to the repo")
end

function load_dataset_splits(dataset_name::AbstractString; repo_root::AbstractString = pwd())
    fold_dir = joinpath(dataset_root(repo_root), dataset_name)
    isdir(fold_dir) || error("missing dataset directory `$fold_dir`")
    train_data = read_split_csv(joinpath(fold_dir, "train.csv"); one_based = false)
    val_data = read_split_csv(joinpath(fold_dir, "val.csv"); one_based = false)
    test_data = read_split_csv(joinpath(fold_dir, "test.csv"); one_based = false)
    n_items = maximum((infer_n(train_data), infer_n(val_data), infer_n(test_data)))
    return (fold_dir = fold_dir, train_data = train_data, val_data = val_data, test_data = test_data, n_items = n_items)
end

function run_dataset_models(dataset_name::AbstractString; d::Int, t::Int, lam::Real, solver_kind::AbstractString = DEFAULT_SOLVER)
    loaded = load_dataset_splits(dataset_name)
    train_data = loaded.train_data
    val_data = loaded.val_data
    test_data = loaded.test_data
    n_items = loaded.n_items

    sos_res = sos_regression(train_data, n_items, d, t, lam; solver_kind = solver_kind)
    poly_res = poly_regression(train_data, n_items, d, lam; solver_kind = solver_kind)
    nec_res = nec_submod_reg(train_data, n_items, d, lam; solver_kind = solver_kind)

    sos_fit = materialize_multilinear_solution(sos_res.wf, sos_res.monoms)
    poly_fit = materialize_poly_solution(poly_res.wf, poly_res.monoms)
    nec_fit = materialize_poly_solution(nec_res.wf, nec_res.monoms)

    rows = [
        regression_row(dataset_name, "sos", sos_res, sos_fit, n_items, train_data, val_data, test_data),
        regression_row(dataset_name, "poly", poly_res, poly_fit, n_items, train_data, val_data, test_data),
        regression_row(dataset_name, "necessary", nec_res, nec_fit, n_items, train_data, val_data, test_data),
    ]

    info = (
        dataset = dataset_name,
        n_items = n_items,
        train_rows = length(train_data),
        val_rows = length(val_data),
        test_rows = length(test_data),
    )
    return (rows = rows, info = info)
end


run_dataset_models (generic function with 1 method)

Three regression models from Section 4.1


In [21]:
function sos_regression(data, n, d, t, lam; solver_kind::AbstractString = DEFAULT_SOLVER)
    model = make_model(; sos = true, solver_kind = solver_kind)

    @polyvar x[1:n]
    monoms = multilinear_monomials(x, d)
    f, wf = build_multilin_var(model, x, monoms, "f")
    f_sos_submod(model, x, f, t)

    m = length(data)
    obj = tot_loss_setwise(f, n, data)
    @objective(model, Min, obj / m + lam * sum(wf[mn]^2 for mn in monoms if mn != 1))
    optimize!(model)
    stats = solver_stats(model)

    return (
        f = f,
        wf = wf,
        x = x,
        obj = value(obj),
        time = stats.run_time,
        monoms = monoms,
        solver = stats.solver,
        termination_status = stats.termination_status,
        primal_status = stats.primal_status,
        dual_status = stats.dual_status,
    )
end

function poly_regression(data, n, d, lam; solver_kind::AbstractString = DEFAULT_SOLVER)
    model = make_model(; sos = false, solver_kind = solver_kind)

    @polyvar x[1:n]
    monoms = monomials(x, 0:d)
    @variable(model, wf[1:length(monoms)])
    @variable(model, dummy_poly)
    @constraint(model, dummy_poly == 0.0)
    f = sum(wf[i] * monoms[i] for i in 1:length(monoms))

    m = length(data)
    A, y = build_feature_matrix(data, monoms, n)
    @expression(model, pred[r = 1:m], sum(A[r, i] * wf[i] for i in 1:length(monoms)))
    @objective(model, Min, sum((pred[r] - y[r])^2 for r in 1:m) / m + lam * sum(wf[i]^2 for i in 2:length(monoms)))
    optimize!(model)
    stats = solver_stats(model)

    return (
        f = f,
        wf = wf,
        x = x,
        obj = objective_value(model),
        time = stats.run_time,
        monoms = monoms,
        solver = stats.solver,
        termination_status = stats.termination_status,
        primal_status = stats.primal_status,
        dual_status = stats.dual_status,
    )
end

function nec_submod_reg(data, n, d, lam; solver_kind::AbstractString = DEFAULT_SOLVER)
    model = make_model(; sos = false, solver_kind = solver_kind)

    @polyvar x[1:n]
    monoms = multilinear_monomials(x, d)
    mon2idx = Dict(m => i for (i, m) in enumerate(monoms))

    @variable(model, wf[1:length(monoms)])
    f = sum(wf[i] * monoms[i] for i in 1:length(monoms))

    m = length(data)
    A, y = build_feature_matrix(data, monoms, n)
    @expression(model, pred[r = 1:m], sum(A[r, i] * wf[i] for i in 1:length(monoms)))
    @objective(model, Min, sum((pred[r] - y[r])^2 for r in 1:m) / m + lam * sum(wf[i]^2 for i in 2:length(monoms)))

    if d >= 4
        len_4 = binomial(n, 4)
        idx_4 = 0
        @variable(model, u4[1:len_4] >= 0)
        for i in 1:n, j in (i + 1):n, l in (j + 1):n, k in (l + 1):n
            idx_4 += 1
            f_ijkl = mon2idx[x[i] * x[j] * x[l] * x[k]]
            @constraint(model, u4[idx_4] >= wf[f_ijkl])
            @constraint(model, u4[idx_4] >= -wf[f_ijkl])

            for quad in ((i, j), (i, l), (i, k), (j, l), (j, k), (l, k))
                @constraint(model, wf[mon2idx[x[quad[1]] * x[quad[2]]]] + u4[idx_4] <= 0)
            end
        end
    end

    triples = d >= 3 ? [(i, j, l) for i in 1:n for j in (i + 1):n for l in (j + 1):n] : Tuple{Int, Int, Int}[]
    pairs = [(i, j) for i in 1:n for j in (i + 1):n]
    pair2trip = Dict{Tuple{Int, Int}, Vector{Int}}(p => Int[] for p in pairs)

    if d >= 3
        for (tidx, (i, j, l)) in enumerate(triples)
            push!(pair2trip[(i, j)], tidx)
            push!(pair2trip[(i, l)], tidx)
            push!(pair2trip[(j, l)], tidx)
        end

        @variable(model, u3[1:length(triples)] >= 0)
        for (tidx, (i, j, l)) in enumerate(triples)
            f_ijl = mon2idx[x[i] * x[j] * x[l]]
            @constraint(model, u3[tidx] >= wf[f_ijl])
            @constraint(model, u3[tidx] >= -wf[f_ijl])
        end

        for (i, j) in pairs
            f_ij = mon2idx[x[i] * x[j]]
            @constraint(model, wf[f_ij] + sum(u3[tidx] for tidx in pair2trip[(i, j)]) <= 0)
        end
    else
        for (i, j) in pairs
            f_ij = mon2idx[x[i] * x[j]]
            @constraint(model, wf[f_ij] <= 0)
        end
    end

    optimize!(model)
    stats = solver_stats(model)

    return (
        f = f,
        wf = wf,
        x = x,
        obj = objective_value(model),
        time = stats.run_time,
        monoms = monoms,
        solver = stats.solver,
        termination_status = stats.termination_status,
        primal_status = stats.primal_status,
        dual_status = stats.dual_status,
    )
end


nec_submod_reg (generic function with 1 method)

To run this, unzip synthetic_data.zip.

In [22]:
demo_datasets = [
    "facility_location_0_noise0.05",
]
demo_d = 2
demo_t = 0
demo_lam = 0.001
solver_kind = DEFAULT_SOLVER

demo_infos = DataFrame([run_dataset_models(name; d = demo_d, t = demo_t, lam = demo_lam, solver_kind = solver_kind).info for name in demo_datasets])
demo_infos


Row,dataset,n_items,train_rows,val_rows,test_rows
,String,Int64,Int64,Int64,Int64
1,facility_location_0_noise0.05,15,1000,500,500


In [23]:
demo_runs = [run_dataset_models(name; d = demo_d, t = demo_t, lam = demo_lam, solver_kind = solver_kind) for name in demo_datasets]
results = DataFrame(vcat([run.rows for run in demo_runs]...))
results

Row,dataset,model,train_mse,val_mse,test_mse,solve_time,solver,termination_status
,String,String,Float64,Float64,Float64,Float64,String,String
1,facility_location_0_noise0.05,sos,0.0479258,0.0469516,0.0404223,0.00128967,SCS,OPTIMAL
2,facility_location_0_noise0.05,poly,0.0460495,0.0503817,0.0406472,0.00126692,SCS,OPTIMAL
3,facility_location_0_noise0.05,necessary,0.407473,0.406281,0.408819,0.00273125,SCS,OPTIMAL


In [24]:
first(CSV.read(joinpath(dataset_root(), demo_datasets[1], "train.csv"), DataFrame), 5)

Row,subset,value
,String,Float64
1,2 3 9,13.1357
2,0 1 2 3 5 6 8 9 10 11 13 14,14.7987
3,5 6 7 8 9 10 12 14,14.3181
4,1 7 14,13.6195
5,4,12.1019
